In [1]:
import numpy as np
print(np.__version__)

2.2.6


In [2]:


class MultiArmedBandit:
    """
    Simulates a k-armed bandit environment.
    True action values q*(a) ~ N(0, 1), actual rewards R ~ N(q*(a), 1).
    """
    def __init__(self, k=10, seed=None):
        self.k = k
        self.rng = np.random.default_rng(seed)
        self.q_star = self.rng.normal(0.0, 1.0, size=k)
        self.optimal_arm = int(np.argmax(self.q_star))

    def pull(self, arm: int) -> float:
        return self.rng.normal(self.q_star[arm], 1.0)


class EpsilonGreedyAgent:
    def __init__(self, k: int, epsilon: float = 0.1):
        self.k = k
        self.epsilon = epsilon
        self.Q = np.zeros(k, dtype=float)
        self.N = np.zeros(k, dtype=int)

    def select_action(self) -> int:
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.k)
        # Break ties randomly among actions with maximum Q
        max_q = np.max(self.Q)
        candidates = np.flatnonzero(self.Q == max_q)
        return int(np.random.choice(candidates))

    def update(self, action: int, reward: float):
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]


class UCBAgent:
    def __init__(self, k: int, c: float = 2.0):
        self.k = k
        self.c = c
        self.Q = np.zeros(k, dtype=float)
        self.N = np.zeros(k, dtype=int)
        self.total_steps = 0

    def select_action(self) -> int:
        self.total_steps += 1
        # Select every untried arm once at the beginning
        untried = np.flatnonzero(self.N == 0)
        if len(untried) > 0:
            return int(untried[0])

        uncertainty = self.c * np.sqrt(np.log(self.total_steps) / self.N)
        ucb_values = self.Q + uncertainty
        max_val = np.max(ucb_values)
        candidates = np.flatnonzero(ucb_values == max_val)
        return int(np.random.choice(candidates))

    def update(self, action: int, reward: float):
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]


class GaussianThompsonSamplingAgent:
    """
    Thompson Sampling for Gaussian rewards with known reward variance sigma^2 = 1.
    Prior for each arm: Normal(mu_0, sigma0^2) = Normal(0, 1).
    """
    def __init__(self, k: int):
        self.k = k
        self.mu_0 = 0.0
        self.var_0 = 1.0
        self.noise_var = 1.0
        
        self.N = np.zeros(k, dtype=int)
        self.sum_r = np.zeros(k, dtype=float)

    def select_action(self) -> int:
        # Compute exact posterior Gaussian parameters for each arm
        post_var = 1.0 / ((1.0 / self.var_0) + (self.N / self.noise_var))
        post_mean = post_var * ((self.mu_0 / self.var_0) + (self.sum_r / self.noise_var))
        
        # Sample one draw from each posterior distribution
        samples = np.random.normal(post_mean, np.sqrt(post_var))
        return int(np.argmax(samples))

    def update(self, action: int, reward: float):
        self.N[action] += 1
        self.sum_r[action] += reward


def evaluate_algorithms(runs: int = 500, steps: int = 1000):
    algorithms = {
        "Greedy (eps=0)": (EpsilonGreedyAgent, {"epsilon": 0.0}),
        "eps-Greedy (eps=0.1)": (EpsilonGreedyAgent, {"epsilon": 0.1}),
        "UCB1 (c=2)": (UCBAgent, {"c": 2.0}),
        "Thompson Sampling": (GaussianThompsonSamplingAgent, {})
    }

    results = {name: {"rewards": np.zeros(steps), "optimal": np.zeros(steps)} 
               for name in algorithms}

    print(f"Running simulation: {runs} independent runs over {steps} steps each...")

    for run_idx in range(runs):
        seed = 10000 + run_idx
        for name, (agent_class, kwargs) in algorithms.items():
            bandit = MultiArmedBandit(k=10, seed=seed)
            agent = agent_class(k=10, **kwargs)

            for t in range(steps):
                action = agent.select_action()
                reward = bandit.pull(action)
                agent.update(action, reward)

                results[name]["rewards"][t] += reward
                if action == bandit.optimal_arm:
                    results[name]["optimal"][t] += 1

    # Average over runs
    for name in algorithms:
        results[name]["rewards"] /= runs
        results[name]["optimal"] /= runs

    return results

if __name__ == "__main__":
    sim_data = evaluate_algorithms(runs=500, steps=1000)

    print("\n--- Summary Performance Across 500 Independent Bandit Tasks ---")
    header = f"{'Algorithm':<22} | {'Avg Reward (Last 100)':<22} | {'Optimal Action % (Step 1000)':<25}"
    print(header)
    print("-" * len(header))
    for name, data in sim_data.items():
        avg_last_100 = np.mean(data["rewards"][-100:])
        opt_final_pct = data["optimal"][-1] * 100.0
        print(f"{name:<22} | {avg_last_100:<22.3f} | {opt_final_pct:<24.1f}%")
    

Running simulation: 500 independent runs over 1000 steps each...

--- Summary Performance Across 500 Independent Bandit Tasks ---
Algorithm              | Avg Reward (Last 100)  | Optimal Action % (Step 1000)
------------------------------------------------------------------------------
Greedy (eps=0)         | 1.040                  | 38.8                    %
eps-Greedy (eps=0.1)   | 1.367                  | 80.0                    %
UCB1 (c=2)             | 1.478                  | 85.4                    %
Thompson Sampling      | 1.513                  | 91.8                    %
